In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

ModuleNotFoundError: No module named 'sklearn.utils'

In [ ]:
# Load the dataset
df = pd.read_csv('Sample - Superstore.csv', encoding='latin-1')

# Show first 5 rows
print("Dataset loaded!")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Keep only the columns we need
df = df[['Order Date', 'Sales']]

# Convert Order Date to proper date format
df['Order Date'] = pd.to_datetime(df['Order Date'])

# Sort by date
df = df.sort_values('Order Date')

# Group sales by month
df['Month'] = df['Order Date'].dt.to_period('M')
monthly_sales = df.groupby('Month')['Sales'].sum().reset_index()
monthly_sales['Month'] = monthly_sales['Month'].dt.to_timestamp()

print("Data cleaned and grouped by month!")
print(f"Total months of data: {len(monthly_sales)}")
monthly_sales.head(10)

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(monthly_sales['Month'], monthly_sales['Sales'], 
         color='steelblue', linewidth=2, marker='o', markersize=4)
plt.title('Monthly Sales Over Time', fontsize=16)
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('historical_sales.png', dpi=150)
plt.show()
print("Chart saved as historical_sales.png")

In [ ]:
# Create numeric features from the date
monthly_sales['Month_Num'] = np.arange(len(monthly_sales))
monthly_sales['Month_of_Year'] = monthly_sales['Month'].dt.month
monthly_sales['Year'] = monthly_sales['Month'].dt.year

print("Features created!")
monthly_sales[['Month', 'Month_Num', 'Month_of_Year', 'Year', 'Sales']].head(10)

In [ ]:
# Use 80% data to train, last 20% to test
split = int(len(monthly_sales) * 0.8)

train = monthly_sales[:split]
test = monthly_sales[split:]

print(f"Training months: {len(train)}")
print(f"Testing months: {len(test)}")

In [ ]:
# Features we use to predict
features = ['Month_Num', 'Month_of_Year', 'Year']

X_train = train[features]
y_train = train['Sales']

X_test = test[features]
y_test = test['Sales']

# Train Linear Regression model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on test data
y_pred = model.predict(X_test)

print("Model trained successfully!")

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("=" * 40)
print("       MODEL PERFORMANCE RESULTS")
print("=" * 40)
print(f"  Mean Absolute Error  (MAE) : ${mae:,.2f}")
print(f"  Root Mean Sq. Error (RMSE) : ${rmse:,.2f}")
print("=" * 40)
print(f"\nThis means our forecast is off by ~${mae:,.0f} on average per month.")

In [ ]:
plt.figure(figsize=(14, 6))

# Plot training data
plt.plot(train['Month'], train['Sales'], 
         label='Historical Sales (Training)', color='steelblue', linewidth=2)

# Plot actual test data
plt.plot(test['Month'], y_test, 
         label='Actual Sales (Test Period)', color='green', 
         linewidth=2, linestyle='--')

# Plot predicted data
plt.plot(test['Month'], y_pred, 
         label='Forecasted Sales', color='red', 
         linewidth=2, linestyle='--', marker='o', markersize=5)

plt.title('Sales Forecast vs Actual Sales', fontsize=16)
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('forecast_vs_actual.png', dpi=150)
plt.show()
print("Chart saved as forecast_vs_actual.png")

In [ ]:
# Generate next 6 months after the last date in dataset
last_month_num = monthly_sales['Month_Num'].max()
last_date = monthly_sales['Month'].max()

future_months = []
for i in range(1, 7):
    next_date = last_date + pd.DateOffset(months=i)
    future_months.append({
        'Month': next_date,
        'Month_Num': last_month_num + i,
        'Month_of_Year': next_date.month,
        'Year': next_date.year
    })

future_df = pd.DataFrame(future_months)
future_df['Forecasted_Sales'] = model.predict(future_df[features])

print("NEXT 6 MONTHS SALES FORECAST")
print("=" * 40)
for _, row in future_df.iterrows():
    print(f"  {row['Month'].strftime('%B %Y')} : ${row['Forecasted_Sales']:,.2f}")
print("=" * 40)

In [ ]:
plt.figure(figsize=(14, 6))

# All historical
plt.plot(monthly_sales['Month'], monthly_sales['Sales'], 
         label='Historical Sales', color='steelblue', linewidth=2)

# Future forecast
plt.plot(future_df['Month'], future_df['Forecasted_Sales'], 
         label='Future Forecast (Next 6 Months)', color='orange', 
         linewidth=2, linestyle='--', marker='o', markersize=7)

# Add value labels on future points
for _, row in future_df.iterrows():
    plt.annotate(f"${row['Forecasted_Sales']:,.0f}", 
                xy=(row['Month'], row['Forecasted_Sales']),
                xytext=(0, 10), textcoords='offset points',
                ha='center', fontsize=8, color='darkorange')

plt.title('Sales Forecast — Next 6 Months', fontsize=16)
plt.xlabel('Month')
plt.ylabel('Total Sales ($)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('future_forecast.png', dpi=150)
plt.show()
print("Chart saved as future_forecast.png")

In [ ]:
avg_forecast = future_df['Forecasted_Sales'].mean()
total_forecast = future_df['Forecasted_Sales'].sum()
best_month = future_df.loc[future_df['Forecasted_Sales'].idxmax()]
slow_month = future_df.loc[future_df['Forecasted_Sales'].idxmin()]

print("=" * 50)
print("        BUSINESS INSIGHTS SUMMARY")
print("=" * 50)
print(f"\n  Forecast Period  : Next 6 Months")
print(f"  Total Revenue    : ${total_forecast:,.2f}")
print(f"  Monthly Average  : ${avg_forecast:,.2f}")
print(f"  Best Month       : {best_month['Month'].strftime('%B %Y')} (${best_month['Forecasted_Sales']:,.2f})")
print(f"  Slowest Month    : {slow_month['Month'].strftime('%B %Y')} (${slow_month['Forecasted_Sales']:,.2f})")
print("\n  RECOMMENDATIONS:")
print(f"  - Stock up inventory before {best_month['Month'].strftime('%B %Y')}")
print(f"  - Run promotions in {slow_month['Month'].strftime('%B %Y')} to boost sales")
print(f"  - Plan staffing for peak demand period")
print("=" * 50)